# SST factorization — QC

Is each fit trustworthy, and how do the fits differ? Interpretation of what the factors
*mean* lives in `02_explore.ipynb`; this notebook only asks whether the models are sound
and comparable.

Reads, for each fit, the two artifacts the runners write:

- `/scratch/sst-drvi/models/<run>/<model>/model.pt` — design, registry and the full
  per-epoch loss history. Read on CPU without constructing the model.
- `/scratch/sst-drvi/embeddings/<run>_<model>_embed.h5ad` — latent space, per-dimension
  stats and the latent UMAP.

CPU-only; everything GPU-bound already happened in `train_drvi.py` / `train_scvi.py`.
Kernel: `sst-drvi`.

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import drvi
import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4))
pd.set_option("display.max_rows", 200)

## Fits under review

`(label, run, model)`. Anything not yet on disk is skipped with a notice rather than
raising, so this notebook is readable while the runs are still going. `drvi 1gpu` is the
original single-GPU baseline, kept as a control on whether moving to 4-GPU DDP — which
quadruples the effective batch to 1024 and disables early stopping — changed the fit.

In [ ]:
FITS = [
    ("drvi", "sst_k64", "drvi"),
    ("drvi+covar", "sst_k64_covar", "drvi"),
    ("scvi+covar", "sst_k64_covar", "scvi"),
    ("drvi 1gpu", "sst_k64_1gpu", "drvi"),
]

models, embeds = {}, {}
for label, run, kind in FITS:
    model_path, embed_path = S.run_paths(run, kind)
    if not (model_path / "model.pt").exists():
        print(f"{label:12s} not fitted yet -> {model_path}")
        continue
    models[label] = model_path
    embeds[label] = ad.read_h5ad(embed_path) if embed_path.exists() else None
    state = "model + embedding" if embeds[label] is not None else "model only (no embedding yet)"
    print(f"{label:12s} {state}")

DRVI_FITS = [l for l, e in embeds.items() if e is not None and models[l].name == "drvi"]


def fit_config(label):
    """The config dict the runner stored in uns, whichever model wrote it."""
    embed = embeds.get(label)
    if embed is None:
        return {}
    for key in ("drvi_config", "scvi_config"):
        if key in embed.uns:
            return json.loads(embed.uns[key])
    return {}

## 1. Model design

What each fit actually is. Rows that do not apply to a model come back as `None` —
`split_method` is DRVI's alone — so the table doubles as a record of where the two model
families differ. `devices` and the effective batch come from the config the runner stored
in `uns`, not from the saved module, because they are training-time rather than
architectural.

In [ ]:
design = S.design_table(models)

extra = {}
for label in design.columns:
    cfg = fit_config(label)
    devices = cfg.get("devices", 1)
    extra[label] = {
        "devices": devices,
        "batch_size (per device)": cfg.get("batch_size"),
        "effective batch": (cfg.get("batch_size") or 0) * devices or None,
        "max_epochs": cfg.get("max_epochs"),
        "kl_warmup_epochs": cfg.get("kl_warmup_epochs", cfg.get("max_epochs")),
    }
design = pd.concat([design, pd.DataFrame(extra)])
design

In [ ]:
# The registry is the audit trail for what was actually registered, as opposed to what
# the command line asked for.
for label, path in models.items():
    print(f"--- {label} ---")
    print(json.dumps(S.model_design(path)["setup_args"], indent=2))

## 2. How are the covariates modelled?

One row per covariate this notebook later associates against, so nothing that gets scored
is missing here. `role` is the important column, and `not modelled` is the important
value: a covariate the model was never told about, which then explains much of a latent
dimension (section 6), means a dimension was spent on nuisance structure. A covariate
marked `batch key` that still scores high means structure leaked past the correction.

`reaches_encoder` is `False` throughout unless a fit was given `--encode-covariates`.
Decoder-only means the reconstruction can source a covariate-driven gene from the
covariate instead of from `z`, which removes the *pressure* on the latent space to encode
it — but a covariate-blind encoder can still put it there. So this table sets up the
question that section 6 answers; it does not settle it.

In [ ]:
CATEGORICAL = ("Supertype", "Brain Region", "Donor ID", "library_prep", "method",
               "Chemistry", "Sex", "Cognitive Status", "Overall AD neuropathological Change",
               "Braak")
CONTINUOUS = ("CPS_Local", "CPS_Global", "CPS_Local_pTau", "CPS_Local_ABeta",
              "Age at Death", "PMI", "Number of UMIs", "Genes detected",
              "Fraction mitochondrial UMIs", "Doublet score")
TESTED = CATEGORICAL + CONTINUOUS

covariates = pd.concat(
    {
        label: S.covariate_design(
            path, TESTED, scaling=(embeds.get(label).uns.get("covariate_scaling")
                                   if embeds.get(label) is not None else None),
        )
        for label, path in models.items()
    },
    axis=1,
)
covariates.loc[:, (slice(None), ["role", "representation", "reaches_encoder"])]

In [ ]:
# The full picture for one fit, level counts and dispersion coupling included.
LABEL = next(iter(models))
print(LABEL)
S.covariate_design(models[LABEL], TESTED,
                   scaling=(embeds[LABEL].uns.get("covariate_scaling")
                            if embeds.get(LABEL) is not None else None))

## 3. Training curves

ELBO and reconstruction loss, train against validation, per fit. Every logged loss
component is stored inside `model.pt`, so these need no retraining.

Epoch 0 dominates the y-range, so the panels start at epoch 1. Watch for the validation
curve peeling away from train (overfitting) and for a curve still descending at the last
epoch (undertrained — and note the DDP fits cannot early-stop, so they always use all
`max_epochs`).

In [ ]:
histories = {label: S.load_history(path) for label, path in models.items()}
LOSSES = ["elbo", "reconstruction_loss"]

fig, axes = plt.subplots(
    len(histories), len(LOSSES),
    figsize=(5.2 * len(LOSSES), 3.4 * len(histories)), squeeze=False,
)
for row, (label, hist) in enumerate(histories.items()):
    tail = hist.iloc[1:]
    for col, loss in enumerate(LOSSES):
        ax = axes[row][col]
        for split, style in (("train", "-"), ("validation", "--")):
            key = f"{loss}_{split}"
            if key in tail:
                ax.plot(tail.index, tail[key], style, label=split, lw=1.4)
        ax.set_title(f"{label} — {loss}", fontsize=10)
        ax.set_xlabel("epoch")
        ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

In [ ]:
rows = {}
for label, hist in histories.items():
    row = {"epochs": len(hist)}
    for loss in LOSSES:
        tr, va = f"{loss}_train", f"{loss}_validation"
        if tr in hist and va in hist:
            row[f"{loss} train"] = round(hist[tr].iloc[-1], 1)
            row[f"{loss} val"] = round(hist[va].iloc[-1], 1)
            row[f"{loss} val-train"] = round(hist[va].iloc[-1] - hist[tr].iloc[-1], 1)
    if "elbo_validation" in hist:
        row["best val epoch"] = int(hist["elbo_validation"].idxmin())
        # still improving at the end => the epoch budget, not convergence, stopped it
        row["val still falling"] = bool(
            hist["elbo_validation"].iloc[-1] <= hist["elbo_validation"].iloc[:-1].min()
        )
    rows[label] = row
pd.DataFrame(rows).T

## 4. How much of `n_latent` did each fit use?

DRVI prunes dimensions it does not need and flags them itself (`var["vanished"]`); scVI has
no such notion. The only way to compare the two is one shared rule — a dimension counts as
used when some cell drives it past `|z| >= 0.5` — so `latent_stats` applies that to both and
carries DRVI's own flag alongside where it exists, rather than trusting either blindly.

A collapsed dimension is a VAE working correctly: with more capacity than the data needs,
the KL term pushes the surplus to the prior, where it contributes nothing to either KL or
reconstruction. So a low count is not by itself a defect — read it against the validation
ELBO in section 3. **A model reaching a lower ELBO on fewer dimensions is the more
parsimonious fit, not the weaker one.**

Two things make the count trustworthy rather than an artifact of the 0.5 cutoff: the std
panel below (a bimodal distribution with a clear gap means any cutoff inside the gap gives
the same answer), and the cumulative variance share (how concentrated the representation
is).

**This is why section 6 reports only live dimensions.** Eta-squared is a variance *ratio*,
so it is scale-invariant and cannot tell a live dimension from one sitting at the prior — a
dimension with std 0.008 that wiggles slightly with library scores as high as a real one.
`factor_association` drops collapsed dimensions and logs how many; without that the heatmap
fills with noise that reads as signal.

In [ ]:
usage = {}
for label, embed in embeds.items():
    if embed is None:
        continue
    stats = S.latent_stats(embed)
    var_share = (stats["std"] ** 2).sort_values(ascending=False)
    var_share = var_share / var_share.sum()
    row = {"n_latent": embed.n_vars, "used (|z|>=0.5)": int(stats["used"].sum())}
    if "vanished" in stats:
        row["not vanished (DRVI flag)"] = int((~stats["vanished"]).sum())
        row["rules disagree"] = int((stats["used"] != ~stats["vanished"]).sum())
    row["median std"] = round(float(stats["std"].median()), 3)
    # how concentrated the representation is: 1.0 in the top 25 means the rest are dead
    for n in (5, 10, 25):
        row[f"var share top{n}"] = round(float(var_share.iloc[:n].sum()), 3)
    usage[label] = row
pd.DataFrame(usage).T

In [ ]:
# Sorted per-dimension std, log scale. A bimodal distribution with a clear gap means the
# "used" count is robust to where the threshold sits; a smooth decline means it is not.
fig, ax = plt.subplots(figsize=(7, 4.2))
for label, embed in embeds.items():
    if embed is None:
        continue
    s = S.latent_stats(embed)["std"].sort_values(ascending=False).to_numpy()
    ax.plot(range(1, len(s) + 1), s, marker="o", ms=3, lw=1.2, label=label)
ax.axhline(0.5, color="grey", ls=":", lw=1)
ax.text(1, 0.52, "|z| threshold", color="grey", fontsize=8, va="bottom")
ax.set_yscale("log")
ax.set_xlabel("latent dimension, ranked by std")
ax.set_ylabel("std across cells (log)")
ax.set_title("Is the dimension count robust to the threshold?")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

In [ ]:
for label in DRVI_FITS:
    print(f"--- {label} ---")
    drvi.utils.pl.plot_latent_dimension_stats(embeds[label], ncols=4, remove_vanished=False)

## 5. Latent UMAPs

Identical covariate lists across fits so the panels compare directly. The check that
matters: `Supertype` and `Brain Region` structure should survive, while `library_prep`,
`method` and the QC metrics should *not* organise the embedding — that would mean the
batch correction failed.

In [ ]:
BIOLOGY = ["Supertype", "Brain Region", "Cognitive Status", "CPS_Local"]
TECHNICAL = ["method", "Chemistry", "Number of UMIs", "Fraction mitochondrial UMIs"]

for label, embed in embeds.items():
    if embed is None:
        continue
    print(f"===== {label}: biology =====")
    sc.pl.umap(embed, color=BIOLOGY, ncols=2, wspace=0.3, size=3)

In [ ]:
for label, embed in embeds.items():
    if embed is None:
        continue
    print(f"===== {label}: technical =====")
    sc.pl.umap(embed, color=TECHNICAL, ncols=2, wspace=0.3, size=3)

In [ ]:
# Donor and library are high-cardinality: no legend, look for block structure.
for label, embed in embeds.items():
    if embed is None:
        continue
    print(f"===== {label}: donor / library =====")
    sc.pl.umap(embed, color=["Donor ID", "library_prep"], ncols=2, wspace=0.1,
               size=3, legend_loc=None)

## 6. Latent dimensions against covariates

Eta-squared for categorical covariates (fraction of the dimension's variance explained by
the grouping) and |Spearman rho| for continuous ones, on one 0-1 scale.

**Read eta-squared with the number of levels in mind.** It rises mechanically with group
count, so `library_prep` (902 levels) and `Donor ID` (84) score high almost regardless of
whether a dimension is really a library artifact, while a 2-level covariate scoring high
(e.g. `Sex`) is unambiguous. Compare a dimension's `library_prep` score against its
`Supertype` score (19 levels) rather than against 1.0.

Collapsed dimensions are excluded (see section 4) — `factor_association` prunes them and
logs the count, so a table shorter than `n_latent` is expected, not a missing row.

In [ ]:
any_embed = next(e for e in embeds.values() if e is not None)
n_levels = pd.Series(
    {col: any_embed.obs[col].nunique() for col in CATEGORICAL},
    name="n_levels",
).sort_values(ascending=False)
print("categorical levels (eta-squared inflates with these):")
print(n_levels.to_string())

assoc = {
    label: S.factor_association(embed, categorical=CATEGORICAL, continuous=CONTINUOUS)
    for label, embed in embeds.items()
    if embed is not None
}
{label: a.shape for label, a in assoc.items()}

In [ ]:
fig, axes = plt.subplots(
    1, len(assoc), figsize=(6.5 * len(assoc), 14), squeeze=False,
)
for ax, (label, a) in zip(axes[0], assoc.items(), strict=True):
    sns.heatmap(a, cmap="rocket_r", vmin=0, vmax=1, linewidths=0.2,
                cbar_kws={"label": "eta$^2$ / |Spearman $\\rho$|"}, ax=ax)
    ax.set_title(label)
    ax.set_xlabel("")
    ax.set_ylabel("latent dimension")
fig.tight_layout()

### The headline comparison

Per covariate, the strongest association any dimension reaches, in each fit — beside how
that fit modelled the covariate. Two readings:

- a covariate that is `not modelled` and scores **high** is a latent dimension spent on
  nuisance structure;
- a covariate that *is* modelled and still scores high means the covariate did not
  actually get absorbed — with decoder-only covariates that is the expected failure mode,
  since the encoder never saw it.

In [ ]:
comparison = pd.DataFrame({label: a.max() for label, a in assoc.items()}).round(3)
roles = pd.DataFrame(
    {label: covariates[(label, "role")] for label in assoc}
).reindex(comparison.index)
comparison = pd.concat({"max association": comparison, "role": roles}, axis=1)
comparison.sort_values(("max association", next(iter(assoc))), ascending=False)

In [ ]:
# Strongest association per dimension, as a biological-vs-technical triage.
# Caveat: `library_prep` (902 levels) and `Donor ID` (84) win on level count alone, so a
# dimension is only convincingly technical when it also beats its own Supertype score.
TECHNICAL_COVS = {"library_prep", "Donor ID", "method", "Chemistry", "Number of UMIs",
                  "Genes detected", "Fraction mitochondrial UMIs", "Doublet score"}
for label, a in assoc.items():
    triage = pd.DataFrame({
        "top_covariate": a.idxmax(axis=1),
        "score": a.max(axis=1).round(3),
        "supertype_score": a["Supertype"].round(3),
    })
    triage["kind"] = np.where(triage["top_covariate"].isin(TECHNICAL_COVS),
                              "technical", "biological")
    triage["margin_over_supertype"] = (triage["score"] - triage["supertype_score"]).round(3)
    print(f"===== {label} =====")
    print(triage["kind"].value_counts().to_string())
    display(triage.sort_values("score", ascending=False).head(12))

## 7. Verdict

Fill in after reading the sections above:

- Fits that are sound enough to interpret:
- Dimensions used, per fit (section 4):
- Any overfitting or undertraining (section 3):
- Did DDP change the fit, judged against `drvi 1gpu`?
- Covariates that are still leaking into the latent space despite being modelled (section 6):
- Covariates that are not modelled and should be:

In [ ]:
from session_info2 import session_info

session_info(dependencies=True)